In [0]:
storage_key = dbutils.secrets.get(scope="kv-finbank", key="storage-account-key")
spark.conf.set("fs.azure.account.key.stfinbankdevfbcq2026.dfs.core.windows.net",storage_key)

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count, min as spark_min, max as spark_max
 
BRONZE_BASE = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net"
SILVER_BASE = "abfss://silver@stfinbankdevfbcq2026.dfs.core.windows.net"
GOLD_BASE = "abfss://gold@stfinbankdevfbcq2026.dfs.core.windows.net"
 
log_ingestas_path = f"{BRONZE_BASE}/_control/log_ingestas"
reporte_calidad_path = f"{SILVER_BASE}/_calidad/reporte_calidad_silver"
REPORTE_DIARIO_PATH = f"{BRONZE_BASE}/_control/reporte_diario_ejecucion"
 
try:
    run_id_actual = dbutils.widgets.get("run_id")
except Exception:
    run_id_actual = "manual"

In [0]:
from pyspark.sql.functions import current_date, to_date
 
df_log_hoy = (
    spark.read.format("delta").load(log_ingestas_path)
    .filter((col("estado") == "Succeeded") & (to_date(col("fecha_ejecucion")) == current_date()))
)
 
bronze_filas = df_log_hoy.agg(spark_sum("filas_copiadas")).collect()[0][0] or 0
bronze_segundos = df_log_hoy.agg(spark_sum("duracion_segundos")).collect()[0][0] or 0
bronze_tablas = df_log_hoy.select("nombre_tabla").distinct().count()
 
print(f"Bronze hoy: {bronze_filas} filas, {bronze_tablas} tablas, {bronze_segundos}s")

In [0]:
df_calidad_hoy = (
    spark.read.format("delta").load(reporte_calidad_path)
    .filter(to_date(col("fecha_ejecucion")) == current_date())
)
 
silver_filas = df_calidad_hoy.agg(spark_sum("total_filas_finales")).collect()[0][0] or 0
silver_rechazados = df_calidad_hoy.agg(spark_sum("registros_rechazados")).collect()[0][0] or 0
 
print(f"Silver hoy: {silver_filas} filas conformes, {silver_rechazados} rechazados")

In [0]:
tablas_gold_principales = [
    "dim_clientes", "fact_transacciones", "fact_cartera", "fact_rentabilidad_cliente"
]
gold_filas = 0
for tabla in tablas_gold_principales:
    try:
        gold_filas += spark.read.format("delta").load(f"{GOLD_BASE}/{tabla}").count()
    except Exception:
        pass
 
print(f"Gold: {gold_filas} filas totales")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, TimestampType, IntegerType, StringType
from datetime import datetime, timezone
 
schema_reporte = StructType([
    StructField("fecha", TimestampType(), False),
    StructField("bronze_filas", IntegerType(), True),
    StructField("bronze_tablas", IntegerType(), True),
    StructField("silver_filas_conformes", IntegerType(), True),
    StructField("silver_rechazados", IntegerType(), True),
    StructField("gold_filas_principales", IntegerType(), True),
    StructField("run_id", StringType(), False),
])
 
fila_reporte = spark.createDataFrame(
    [Row(
        fecha=datetime.now(timezone.utc),
        bronze_filas=int(bronze_filas),
        bronze_tablas=int(bronze_tablas),
        silver_filas_conformes=int(silver_filas),
        silver_rechazados=int(silver_rechazados),
        gold_filas_principales=int(gold_filas),
        run_id=run_id_actual,
    )],
    schema=schema_reporte
)
fila_reporte.write.format("delta").mode("append").option("mergeSchema", "true").save(REPORTE_DIARIO_PATH)
 
print(f"\nRESUMEN DIARIO DE EJECUCION")
print(f"Bronze: {bronze_filas} filas procesadas en {bronze_tablas} tablas")
print(f"Silver: {silver_filas} filas conformes, {silver_rechazados} rechazadas por calidad")
print(f"Gold: {gold_filas} filas en tablas principales")
 
dbutils.notebook.exit("Reporte diario generado")